In [31]:
import sys
import os

CURRENT_DIR = os.getcwd()

if os.path.basename(CURRENT_DIR) == "notebooks":
    PROJECT_ROOT = os.path.abspath("..")
else:
    PROJECT_ROOT = CURRENT_DIR

if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

from src.preprocess import preprocess_query

## 1.Import thư viện

In [ ]:
import os
import math
import pickle
import pandas as pd
import numpy as np

from collections import Counter
from scipy.sparse import csr_matrix

from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import cosine_similarity

## 2.Khai báo đường dẫn

In [2]:
DATA_PATH = "../data/processed/news_processed.pkl"

MODEL_DIR = "../models/lsi_svd"
os.makedirs(MODEL_DIR, exist_ok=True)

## 3.Load dữ liệu đã xử lý

In [3]:
df = pd.read_pickle(DATA_PATH)

print("Shape:", df.shape)
print(df.columns.tolist())

df[["doc_id", "title", "topic", "source", "processed_text"]].head()

Shape: (168169, 11)
['id', 'title', 'content', 'topic', 'source', 'url', 'raw_text', 'processed_text', 'tokens', 'token_count', 'doc_id']


,doc_id,title,topic,source,processed_text
0,0,"Tên cướp tiệm vàng tại Huế là đại uý công an, ...",Pháp luật,docbao.vn,tên cướp tiệm vàng huế đại_úy công_an công_tác...
1,1,"Bỏ qua mạng 5G, Nga tiến thẳng từ 4G lên 6G",Sống kết nối,vtc.vn,bỏ_qua mạng 5 g nga tiến thẳng 4 g lên 6 g bỏ_...
2,2,Địa phương nào đứng đầu cả nước tổng điểm 3 mô...,Giáo dục,thanhnien.vn,địa_phương nào đứng đầu cả nước tổng_điểm 3 mô...
3,3,Người chết trong mưa lũ 'nghìn năm có một' ở M...,Thế giới,vnexpress,người chết mưa_lũ nghìn năm mỹ tăng lên 28 ngư...
4,4,"Hải Phòng: Hình ảnh xe ""điên"" gây tai nạn liên...",Thời sự - Xã hội,soha,hải_phòng hình_ảnh xe điên gây tai_nạn liên_ho...


## 4.Lấy documents đầu vào

In [4]:
documents = df["processed_text"].fillna("").astype(str).tolist()

print("Số lượng bài báo:", len(documents))

Số lượng bài báo: 168169


## 5.Tạo TF-IDF Matrix

In [33]:
tokenized_docs = [doc.split() for doc in documents]

N = len(tokenized_docs)

min_df = 5
max_df_ratio = 0.85

# 1. Tính Document Frequency: mỗi từ xuất hiện trong bao nhiêu document
df_counter = Counter()

for tokens in tokenized_docs:
    df_counter.update(set(tokens))

max_df = int(max_df_ratio * N)

# 2. Lọc từ quá hiếm và quá phổ biến
valid_tokens = {
    token: df_count
    for token, df_count in df_counter.items()
    if df_count >= min_df and df_count <= max_df
}

# 3. Không giới hạn max_features
# Lấy toàn bộ token hợp lệ sau lọc
valid_tokens_sorted = sorted(
    valid_tokens.items(),
    key=lambda x: x[1],
    reverse=True
)

vocab = {
    token: idx
    for idx, (token, _) in enumerate(valid_tokens_sorted)
}

print("Số lượng documents:", N)
print("Tổng số token ban đầu:", len(df_counter))
print("Vocabulary size sau lọc:", len(vocab))

# 4. Tính IDF
idf = np.zeros(len(vocab), dtype=np.float32)

for token, idx in vocab.items():
    df_t = df_counter[token]
    idf[idx] = math.log((N + 1) / (df_t + 1)) + 1

# 5. Tính TF-IDF Matrix
rows = []
cols = []
values = []

for doc_idx, tokens in enumerate(tokenized_docs):
    token_counts = Counter(tokens)

    for token, count in token_counts.items():
        if token in vocab:
            col_idx = vocab[token]

            # Sublinear TF
            tf = 1 + math.log(count)

            # TF-IDF = TF * IDF
            tfidf_value = tf * idf[col_idx]

            rows.append(doc_idx)
            cols.append(col_idx)
            values.append(tfidf_value)

tfidf_matrix = csr_matrix(
    (values, (rows, cols)),
    shape=(N, len(vocab)),
    dtype=np.float32
)

# 6. Chuẩn hóa L2 để tính cosine similarity tốt hơn
tfidf_matrix = normalize(tfidf_matrix, norm="l2", axis=1)

print("TF-IDF Matrix:", tfidf_matrix.shape)
print("Non-zero values:", tfidf_matrix.nnz)

Số lượng documents: 168169
Tổng số token ban đầu: 515623
Vocabulary size sau lọc: 99193
TF-IDF Matrix: (168169, 99193)
Non-zero values: 30506165


## 6.Dùng SVD giảm chiều

In [34]:
candidate_components = [100, 200, 300, 400, 500, 600, 700, 800]

svd_results = []

for n in candidate_components:
    svd_temp = TruncatedSVD(
        n_components=n,
        random_state=42
    )

    svd_temp.fit(tfidf_matrix)

    explained_variance = svd_temp.explained_variance_ratio_.sum()

    svd_results.append({
        "n_components": n,
        "explained_variance": explained_variance
    })

    print(f"n_components = {n}, explained_variance = {explained_variance:.4f}")

n_components = 100, explained_variance = 0.1668
n_components = 200, explained_variance = 0.2164
n_components = 300, explained_variance = 0.2521
n_components = 400, explained_variance = 0.2816
n_components = 500, explained_variance = 0.3074
n_components = 600, explained_variance = 0.3306
n_components = 700, explained_variance = 0.3517
n_components = 800, explained_variance = 0.3712


In [35]:
n_components = 500

svd_model = TruncatedSVD(
    n_components=n_components,
    random_state=42
)

lsi_doc_matrix = svd_model.fit_transform(tfidf_matrix)

print("LSI Matrix:", lsi_doc_matrix.shape)
print("n_components:", n_components)
print("Explained variance:", svd_model.explained_variance_ratio_.sum())

LSI Matrix: (168169, 500)
n_components: 500
Explained variance: 0.30735755


## 7.Chuẩn hóa vector LSI

In [36]:
lsi_doc_matrix = normalize(lsi_doc_matrix, norm="l2")

print("LSI Matrix after normalize:", lsi_doc_matrix.shape)

LSI Matrix after normalize: (168169, 500)


## 8.Tạo metadata để hiển thị kết quả

In [37]:
metadata = df[[
    "doc_id",
    "id",
    "title",
    "content",
    "topic",
    "source",
    "url"
]].copy()

metadata.head()

,doc_id,id,title,content,topic,source,url
0,0,218270,"Tên cướp tiệm vàng tại Huế là đại uý công an, ...","Chiều 31/7, Công an tỉnh Thừa Thiên - Huế đã c...",Pháp luật,docbao.vn,https://docbao.vn/phap-luat/ten-cuop-tiem-vang...
1,1,218269,"Bỏ qua mạng 5G, Nga tiến thẳng từ 4G lên 6G","Gần đây, Thứ trưởng Bộ Phát triển Kỹ thuật số,...",Sống kết nối,vtc.vn,https://vtc.vn/bo-qua-mang-5g-nga-tien-thang-t...
2,2,218268,Địa phương nào đứng đầu cả nước tổng điểm 3 mô...,Kết quả thi tốt nghiệp THPT năm 2022 cho thấy ...,Giáo dục,thanhnien.vn,https://thanhnien.vn/dia-phuong-nao-dung-dau-c...
3,3,218267,Người chết trong mưa lũ 'nghìn năm có một' ở M...,Thống đốc Kentucky Andy Beshear hôm 31/7 cho h...,Thế giới,vnexpress,https://vnexpress.net/nguoi-chet-trong-mua-lu-...
4,4,218266,"Hải Phòng: Hình ảnh xe ""điên"" gây tai nạn liên...",Vụ tai nạn giao thông liên hoàn trên phố đi bộ...,Thời sự - Xã hội,soha,https://soha.vn/hai-phong-hinh-anh-xe-dien-gay...


## 9.Lưu model/index

In [38]:
with open(f"{MODEL_DIR}/vocab.pkl", "wb") as f:
    pickle.dump(vocab, f)

with open(f"{MODEL_DIR}/idf.pkl", "wb") as f:
    pickle.dump(idf, f)

with open(f"{MODEL_DIR}/svd_model.pkl", "wb") as f:
    pickle.dump(svd_model, f)

with open(f"{MODEL_DIR}/lsi_doc_matrix.pkl", "wb") as f:
    pickle.dump(lsi_doc_matrix, f)

with open(f"{MODEL_DIR}/metadata.pkl", "wb") as f:
    pickle.dump(metadata, f)

print("Đã lưu model vào:", MODEL_DIR)

Đã lưu model vào: ../models/lsi_svd


## 10.Hàm xử lý query

In [ ]:
def query_to_tfidf(query_tokens):
    token_counts = Counter(query_tokens)

    rows = []
    cols = []
    values = []

    for token, count in token_counts.items():
        if token in vocab:
            col_idx = vocab[token]

            tf = 1 + math.log(count)
            tfidf_value = tf * idf[col_idx]

            rows.append(0)
            cols.append(col_idx)
            values.append(tfidf_value)

    query_vector = csr_matrix(
        (values, (rows, cols)),
        shape=(1, len(vocab)),
        dtype=np.float32
    )

    query_vector = normalize(query_vector, norm="l2", axis=1)

    return query_vector

## 11. Hàm tìm kiếm bằng TF-IDF gốc

Cell này dùng trực tiếp ma trận **TF-IDF gốc** để truy xuất tài liệu. Query được xử lý giống document, sau đó chuyển thành vector TF-IDF thủ công và tính **cosine similarity** với toàn bộ document.

In [ ]:
def search_tfidf(query, top_k=10):
    # 1. Xử lý query giống dữ liệu document
    query_processed, query_tokens = preprocess_query(query)

    # 2. Query tokens -> TF-IDF vector thủ công
    query_tfidf = query_to_tfidf(query_tokens)

    # 3. Tính cosine similarity trong không gian TF-IDF gốc
    scores = cosine_similarity(query_tfidf, tfidf_matrix).flatten()

    # 4. Lấy top-k document có điểm cao nhất
    top_indices = scores.argsort()[::-1][:top_k]

    # 5. Lấy thông tin bài báo tương ứng
    results = metadata.iloc[top_indices].copy()
    results["score"] = scores[top_indices]
    results["query_processed"] = query_processed

    return results

## 12. Hàm tìm kiếm bằng LSI/SVD

Cell này truy xuất tài liệu bằng phương pháp **LSI/SVD**. Query được chuyển sang TF-IDF thủ công, sau đó đưa qua mô hình SVD để vào cùng không gian LSI với document, rồi tính cosine similarity.

In [ ]:
def search_lsi(query, top_k=10):
    # 1. Xử lý query giống dữ liệu document
    query_processed, query_tokens = preprocess_query(query)

    # 2. Query tokens -> TF-IDF vector thủ công
    query_tfidf = query_to_tfidf(query_tokens)

    # 3. Query TF-IDF -> không gian LSI/SVD
    query_lsi = svd_model.transform(query_tfidf)
    query_lsi = normalize(query_lsi, norm="l2")

    # 4. Tính cosine similarity trong không gian LSI
    scores = cosine_similarity(query_lsi, lsi_doc_matrix).flatten()

    # 5. Lấy top-k document có điểm cao nhất
    top_indices = scores.argsort()[::-1][:top_k]

    # 6. Lấy thông tin bài báo tương ứng
    results = metadata.iloc[top_indices].copy()
    results["score"] = scores[top_indices]
    results["query_processed"] = query_processed

    return results

## 13. Test riêng phương pháp TF-IDF gốc

Cell này chạy thử truy vấn bằng **TF-IDF gốc** để xem các kết quả được xếp hạng trực tiếp theo vector TF-IDF.

In [ ]:
query = "cầu thủ Quang Hải"

tfidf_results = search_tfidf(query, top_k=10)

print("Query gốc:", query)
print("Query sau xử lý:", tfidf_results["query_processed"].iloc[0])

tfidf_results[[
    "doc_id",
    "title",
    "topic",
    "source",
    "url",
    "score"
]]

## 14. Test riêng phương pháp LSI/SVD

Cell này chạy thử cùng một truy vấn bằng **LSI/SVD** để xem kết quả sau khi document và query được đưa vào không gian ngữ nghĩa ẩn.

In [ ]:
query = "cầu thủ Quang Hải"

lsi_results = search_lsi(query, top_k=10)

print("Query gốc:", query)
print("Query sau xử lý:", lsi_results["query_processed"].iloc[0])

lsi_results[[
    "doc_id",
    "title",
    "topic",
    "source",
    "url",
    "score"
]]

## 15. So sánh TF-IDF gốc và LSI/SVD trên cùng query

Cell này chạy cùng một query trên cả hai phương pháp và hiển thị kết quả cạnh nhau. Mục tiêu là so sánh sự khác biệt giữa truy xuất theo **từ khóa TF-IDF** và truy xuất theo **không gian ngữ nghĩa LSI/SVD**.

In [ ]:
def compare_tfidf_lsi(query, top_k=10):
    tfidf_results = search_tfidf(query, top_k=top_k).reset_index(drop=True)
    lsi_results = search_lsi(query, top_k=top_k).reset_index(drop=True)

    compare_df = pd.DataFrame({
        "rank": range(1, top_k + 1),
        "tfidf_doc_id": tfidf_results["doc_id"],
        "tfidf_title": tfidf_results["title"],
        "tfidf_score": tfidf_results["score"],
        "lsi_doc_id": lsi_results["doc_id"],
        "lsi_title": lsi_results["title"],
        "lsi_score": lsi_results["score"],
    })

    return compare_df

query = "cầu thủ Quang Hải"

print("Query gốc:", query)
print("Query sau xử lý:", preprocess_query(query)[0])

compare_tfidf_lsi(query, top_k=10)